[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/04-embeddings-and-vector-databases/code/embeddings_and_vector_db.ipynb)

# Class 4.4: Embeddings and vector databases

Give the model a searchable memory of real documents. Our corpus is a set of actual IRS tax publications (public domain), chunked and tagged with metadata (publication, tax year, page, source URL). We embed the chunks, index them with FAISS, and query by meaning, then add metadata filtering and a hybrid keyword blend. This retriever is reused in class 4.5 (RAG from scratch).

**What we will cover:** chunking, embeddings and cosine similarity, a FAISS index, metadata filtering (drop last year's tax figures), and hybrid search.

Chunking and BM25 run anywhere; the embedding cells download a small model on first run (Colab or any machine with internet). The corpus ships with this notebook in `data/corpus.jsonl`; `data/fetch_corpus.py` rebuilds the full set from the original PDFs.

## Setup

New libraries for this class:

```
pip install sentence-transformers faiss-cpu rank-bm25
```

## 1. Chunking: split a long page into passages

A real publication page is a few hundred words, too long to embed as one vector. We split it into overlapping word windows. The overlap keeps a fact that straddles a boundary from being cut in half.

In [1]:
# A long page is too big to embed as one vector, so we cut it into overlapping
# windows of `size` words. `overlap` repeats a few words across the boundary so a
# fact sitting on the seam is not split and lost.
def chunk_text(text, size=40, overlap=10):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(' '.join(words[i:i + size]))     # take a window of `size` words
        if i + size >= len(words):
            break                                      # reached the end
        i += size - overlap                            # step forward, minus the overlap
    return chunks

# Stand in for one raw page: the Pub 502 (medical expenses) passages joined together.
import json
rows = [json.loads(l) for l in open('data/corpus.jsonl')]   # each line is one record
page = ' '.join(r['text'] for r in rows if r['pub'] == 'Pub 502')
pieces = chunk_text(page, size=40, overlap=10)
print('page word count:', len(page.split()))
print('chunks produced :', len(pieces))
# Sanity check the overlap: last 10 words of chunk 0 equal the first 10 of chunk 1.
print('overlap check   :', pieces[0].split()[-10:] == pieces[1].split()[:10])
# -> page word count: 201 | chunks produced: 7 | overlap check: True

page word count: 201
chunks produced : 7
overlap check   : True


In [2]:
for p in pieces:
    print('chunk word count:', len(p.split()), ' | ', p[:60], '...')

chunk word count: 40  |  Deducting medical and dental expenses. You can deduct on Sch ...
chunk word count: 40  |  gross income. You must itemize deductions to claim them, and ...
chunk word count: 40  |  medical expenses the amounts you pay for dental treatment, i ...
chunk word count: 40  |  procedures are not deductible. What you can include. Deducti ...
chunk word count: 40  |  lenses, hearing aids, and premiums you pay for medical insur ...
chunk word count: 40  |  cannot include. You generally cannot deduct nonprescription  ...
chunk word count: 21  |  insurance or paid through a tax-advantaged account. Everyday ...


## 2. The corpus and its metadata

Each record is a real passage from an IRS publication with metadata attached. The metadata is what makes filtering and citations possible later. Notice there are two tax years for the standard deduction: last year's figure is stale and we will filter it out in section 5.

In [3]:
texts = [r['text'] for r in rows]
print(len(rows), 'chunks loaded')
print('publications:', sorted({r['pub'] for r in rows}))
print('tax years   :', sorted({r['tax_year'] for r in rows}))
print()
sample = rows[0]
for key in ('id', 'pub', 'tax_year', 'page', 'source_url'):
    print(f'{key:11}: {sample[key]}')
print('text       :', sample['text'][:80], '...')
# -> 19 chunks loaded; publications Pub 501/502/526/936/969; tax years [2024, 2025]

19 chunks loaded
publications: ['Pub 501', 'Pub 502', 'Pub 526', 'Pub 936', 'Pub 969']
tax years   : [2024, 2025]

id         : p501-2024-stdded
pub        : Pub 501
tax_year   : 2024
page       : 24
source_url : https://www.irs.gov/pub/irs-prior/p501--2024.pdf
text       : Standard deduction for 2024. For tax year 2024 the basic standard deduction is 1 ...


## 3. Embeddings and cosine similarity

An embedding turns a passage into a vector so that passages with similar meaning land near each other. We test that with an everyday-worded question that shares almost no words with the target passage. Keyword search would struggle; meaning search should not.

In [ ]:
import logging
# Show INFO logs (llm.py logs every model call; we add our own below).
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
log = logging.getLogger("course.rag")
# An embedding turns text into a vector of numbers so that similar MEANINGS land
# near each other, even when the words differ.
from sentence_transformers import SentenceTransformer
import numpy as np
import pathlib

model = SentenceTransformer('all-MiniLM-L6-v2')          # small model, downloads once

# Embedding every passage is the slow step, so we cache it. If
# data/precompute_embeddings.py has been run, data/embeddings.npy and data/ids.json
# already exist, so we load them and skip re-embedding. This is the SAME whether
# corpus.jsonl is the shipped sample or was rebuilt from the real PDFs by
# data/fetch_corpus.py. If the cache is missing or does not match the corpus, we
# embed now and write it, so the next run is instant.

emb_path, ids_path = pathlib.Path('data/embeddings.npy'), pathlib.Path('data/ids.json')
ids_now = [r['id'] for r in rows]
if emb_path.exists() and ids_path.exists() and json.loads(ids_path.read_text()) == ids_now:
    emb = np.load(emb_path).astype('float32')            # cached vectors, instant
    print('loaded cached embeddings:', emb.shape)        # [19, 384]
else:
    emb = np.array(model.encode(texts, normalize_embeddings=True), dtype='float32')
    np.save(emb_path, emb); ids_path.write_text(json.dumps(ids_now))
    print('embedded and cached    :', emb.shape)         # one 384-length vector per chunk

# Cosine similarity measures the ANGLE between two vectors: 1.0 = same direction
# (very similar), 0 = unrelated. a @ b is the dot product.
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

by_id = {r['id']: emb[i] for i, r in enumerate(rows)}    # look up a chunk's vector by id
# An everyday-worded question that shares almost no words with the target passage:
q = model.encode('writing off what I paid to fix my kids teeth',
                 normalize_embeddings=True)

print('vs dental passage      :', round(cosine(q, by_id['p502-2025-dental']), 3))
print('vs mortgage points     :', round(cosine(q, by_id['p936-2025-points']), 3))

# Expected (real model): the dental passage scores clearly higher, though the
# query shares no words like 'deduct' or 'dental' with it. That is meaning search.
# (In section 6 we will see plain keyword search rank a wrong passage first here.)

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded cached embeddings: (19, 384)


NameError: name 'model' is not defined

## 4. A FAISS index for fast nearest-neighbor search

FAISS stores the vectors and finds the nearest ones to a query fast. On normalized vectors, inner product equals cosine similarity. We return the metadata with each hit, which is exactly what we will cite in class 4.5 (RAG from scratch).

In [ ]:
# FAISS stores many vectors and finds the nearest ones to a query fast. We use
# IndexFlatIP (inner product). Because our vectors are normalized to length 1,
# inner product equals cosine similarity, so "largest inner product" = "most similar".
import faiss

vectors = np.array(emb, dtype='float32')      # FAISS wants a float32 array
index = faiss.IndexFlatIP(vectors.shape[1])   # shape[1] is the vector length (384)
index.add(vectors)                            # load all chunk vectors into the index

def search(query, k=3):
    # Encode the query and reshape to a 1-row matrix ([None, :]) as FAISS expects.
    qv = np.array(model.encode(query, normalize_embeddings=True), dtype='float32')[None, :]
    scores, ids = index.search(qv, k)         # top-k: similarity scores and row indices
    hits = []
    for s, i in zip(scores[0], ids[0]):       # [0] because we searched one query
        r = rows[i]                           # map the row index back to its record
        hits.append((r['id'], round(float(s), 3), f"{r['pub']} {r['tax_year']} p{r['page']}"))
    log.info("search query=%r -> %s", query, [h[0] for h in hits])
    return hits

for hit in search('how much of my medical costs can I deduct', k=3):
    print(hit)
# Expected (real model): the Pub 502 medical-expense passages (threshold and what
# you can include) are the top hits.

## 5. Metadata filtering: drop last year's tax figures

The corpus holds the standard deduction for two tax years. Ask for it and both come back, but last year's number is wrong for a current return. FAISS stores only vectors, so we keep the records alongside and filter after the search. (A full vector database, class 4.6, filters during the search.)

In [ ]:
# Metadata filtering: FAISS only knows vectors, so to filter by a field (here the
# tax year) we OVER-FETCH a larger pool, then keep only the rows that match.
def search_filtered(query, tax_year=None, k=3, pool=19):
    qv = np.array(model.encode(query, normalize_embeddings=True), dtype='float32')[None, :]
    scores, ids = index.search(qv, pool)          # fetch a wide pool first
    hits = []
    for s, i in zip(scores[0], ids[0]):
        r = rows[i]
        if tax_year is not None and r['tax_year'] != tax_year:
            continue                              # skip rows that fail the filter
        hits.append((r['id'], r['tax_year']))
        if len(hits) == k:                        # stop once we have k matches
            break
    return hits

print('no filter :', search_filtered('standard deduction for a single filer', k=4))
print('2025 only :', search_filtered('standard deduction for a single filer', tax_year=2025, k=3))
# Expected: the unfiltered results include p501-2024-stdded (last year's 14,600
# figure); the 2025 filter drops it, so only the current 15,000 passage remains.
# The filter is plain Python on the metadata, so this part is deterministic.

## 6. Hybrid search: meaning plus exact keywords

Dense search captures meaning but can miss an exact term; keyword search (BM25) catches exact terms but misses paraphrases. Blending both is more robust. Watch keyword search alone rank a wrong passage first on the reworded dental question, then see the blend recover.

In [ ]:
# Hybrid search blends two signals: dense (meaning) and BM25 (exact keywords).
from rank_bm25 import BM25Okapi

# BM25 is a classic keyword-relevance score. It needs the corpus pre-tokenized.
bm25 = BM25Okapi([t.lower().split() for t in texts])

# The two scores live on different scales, so we rescale each to 0..1 before mixing.
def minmax(x):
    x = np.array(x, dtype=float); span = x.max() - x.min()
    return (x - x.min()) / span if span > 0 else np.zeros_like(x)

# Sort scores high-to-low and return the top-k as (id, score). argsort gives
# ascending order, so [::-1] reverses it to descending.
def ranked_ids(scores, k=3):
    order = np.argsort(scores)[::-1][:k]
    return [(rows[i]['id'], round(float(scores[i]), 3)) for i in order]

query = 'writing off what I paid to fix my kids teeth'
key = bm25.get_scores(query.lower().split())                 # keyword score per chunk
vec = emb @ model.encode(query, normalize_embeddings=True)   # cosine (vectors normalized)
hybrid = 0.5 * minmax(vec) + 0.5 * minmax(key)               # equal blend of both

print('keyword only:', ranked_ids(key))
print('hybrid      :', ranked_ids(hybrid))
# keyword only (deterministic): p502-2025-notdeductible tops the list, the WRONG
# passage, because it repeats common words. Expected with the real model: the dense
# score lifts p502-2025-dental, and the hybrid puts it on top.

## Recap

You chunked a real publication page, embedded the passages, and searched by meaning with a FAISS index, then filtered out last year's tax figures by metadata and blended in BM25 keyword scores. This is the retriever half of RAG. In class 4.5 (RAG from scratch) we feed these retrieved passages to the model and answer questions from the documents, with citations back to the publication and page.